# Evaluate DDPG on Fixed Obstacles

Load the saved DDPG checkpoint and evaluate it on a fixed unseen-obstacle configuration.

## Imports and Paths

In [1]:
from pathlib import Path
import sys
import torch
from IPython.display import Video, display

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()
    if BASE_DIR.name != "Continuous_Diff_Drive":
        BASE_DIR = BASE_DIR / "Continuous_Diff_Drive"

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from diff_drive_agent import DiffDriveDDPGAgent
from diff_drive_env import DiffDriveEnv

BASE_DIR

PosixPath('/Users/anthonymc/Desktop/NAML_project/Continuous_Diff_Drive')

## Fixed Evaluation Configuration

In [2]:
CHECKPOINT_PATH = BASE_DIR / "models" / "ddpg_checkpoint.pt"
EVALUATION_VIDEO_DIR = BASE_DIR / "videos" / "evaluation_ddpg"
EVALUATION_NAME_PREFIX_FIXED = "ddpg_diff_drive_eval_fixed_obstacles"
EVALUATION_NAME_PREFIX_RANDOM = "ddpg_diff_drive_eval_random_obstacles"

# A room with a few axis-aligned rectangular obstacles.
# Each obstacle is (x, y, width, height) in metres, origin at bottom-left.
OBSTACLES = [
    (2.0, 4.0, 3.0, 0.3),
    (5.0, 2.0, 0.3, 3.0),
    (7.0, 6.0, 1.5, 0.3),
]

ENV_KWARGS = dict(
    room_size       = (10.0, 10.0),
    obstacles       = OBSTACLES,
    random_obst     = False,
    robot_start     = (1.0, 1.0),
    goal_pos        = (8.5, 8.5),
    max_step        = 1000,
    n_lidar_rays    = 16,
    lidar_max_range = 5.0,
    robot_radius    = 0.3,
    dt              = 0.1,
    render_mode     = "rgb_array",
    obstacle_mode   = "curriculum",
)

ENV_KWARGS_RAND = dict(
    room_size       = (10.0, 10.0),
    obstacles       = None,
    random_obst     = True,
    robot_start     = (1.0, 1.0),
    goal_pos        = (8.5, 8.5),
    max_step        = 1000,
    n_lidar_rays    = 16,
    lidar_max_range = 5.0,
    robot_radius    = 0.3,
    dt              = 0.1,
    render_mode     = "rgb_array",
    obstacle_mode   = "curriculum",
)

N_EPISODES = 3

EVALUATION_VIDEO_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH
DEVICE = "cpu"

## Environment and Agent

In [3]:
env = DiffDriveEnv(**ENV_KWARGS)
agent = DiffDriveDDPGAgent(env)

env_rand = DiffDriveEnv(**ENV_KWARGS_RAND)
agent_rand = DiffDriveDDPGAgent(env_rand)

## Load Checkpoint

In [4]:
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"No checkpoint found at {CHECKPOINT_PATH}")

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
agent.actor.load_state_dict(checkpoint["actor"])
agent.critic.load_state_dict(checkpoint["critic"])
agent.actor_target.load_state_dict(checkpoint["actor_target"])
agent.critic_target.load_state_dict(checkpoint["critic_target"])

print(f"Loaded checkpoint from {CHECKPOINT_PATH}")

agent_rand.actor.load_state_dict(checkpoint["actor"])
agent_rand.critic.load_state_dict(checkpoint["critic"])
agent_rand.actor_target.load_state_dict(checkpoint["actor_target"])
agent_rand.critic_target.load_state_dict(checkpoint["critic_target"])

print(f"Loaded checkpoint from {CHECKPOINT_PATH}")

Loaded checkpoint from /Users/anthonymc/Desktop/NAML_project/Continuous_Diff_Drive/models/ddpg_checkpoint.pt
Loaded checkpoint from /Users/anthonymc/Desktop/NAML_project/Continuous_Diff_Drive/models/ddpg_checkpoint.pt


## Greedy Evaluation

In [5]:
agent.eval_recorded(
    video_folder = EVALUATION_VIDEO_DIR,
    name_prefix  = EVALUATION_NAME_PREFIX_FIXED,
    n_episodes   = N_EPISODES,
    add_noise    = False
)

# Display the fixed-obstacle videos first
video_paths = sorted(EVALUATION_VIDEO_DIR.glob(f"{EVALUATION_NAME_PREFIX_FIXED}_*.mp4"))
if not video_paths:
    print(f"No fixed-obstacle videos found yet in {EVALUATION_VIDEO_DIR}")
for video_path in video_paths:
    print(video_path.name)
    display(Video(filename=str(video_path), embed=True))

/Users/anthonymc/Desktop/NAML_project/venv/lib/python3.11/site-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /Users/anthonymc/Desktop/NAML_project/Continuous_Diff_Drive/videos/evaluation_ddpg folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


  Eval ep 1: reward = 165.7 | steps = 124 | goal reached: ✓
  Eval ep 2: reward = 165.7 | steps = 124 | goal reached: ✓
  Eval ep 3: reward = 165.7 | steps = 124 | goal reached: ✓
No fixed-obstacle videos found yet in /Users/anthonymc/Desktop/NAML_project/Continuous_Diff_Drive/videos/evaluation_ddpg


## Random Obstacle evaluation

In [6]:
agent_rand.eval_recorded(
    video_folder = EVALUATION_VIDEO_DIR,
    name_prefix  = EVALUATION_NAME_PREFIX_RANDOM,
    n_episodes   = N_EPISODES,
    add_noise    = False
)

# Display the random-obstacle videos as well
video_paths_rand = sorted(EVALUATION_VIDEO_DIR.glob(f"{EVALUATION_NAME_PREFIX_RANDOM}_*.mp4"))
if not video_paths_rand:
    print(f"No random-obstacle videos found yet in {EVALUATION_VIDEO_DIR}")
for video_path in video_paths_rand:
    print(video_path.name)
    display(Video(filename=str(video_path), embed=True))

/Users/anthonymc/Desktop/NAML_project/venv/lib/python3.11/site-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /Users/anthonymc/Desktop/NAML_project/Continuous_Diff_Drive/videos/evaluation_ddpg folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


  Eval ep 1: reward = 176.3 | steps = 106 | goal reached: ✓
  Eval ep 2: reward = 167.2 | steps = 124 | goal reached: ✓
  Eval ep 3: reward = 176.7 | steps = 104 | goal reached: ✓
No random-obstacle videos found yet in /Users/anthonymc/Desktop/NAML_project/Continuous_Diff_Drive/videos/evaluation_ddpg
